# Experimento — Tabla maestra con datos reales de USDA
TFM: Sistema inteligente de gestión alimentaria — Fase II (sección 4.3)

## Celda 1 — Configurar tu API key de forma segura

En Colab: icono de llave 🔑 en el panel izquierdo → **Secrets** → **Add new secret** → nombre: `USDA_API_KEY` → valor: tu clave → activa **Notebook access**.

In [1]:
from google.colab import userdata

try:
    USDA_API_KEY = userdata.get('USDA_API_KEY')
    print("✅ API key cargada correctamente desde Secrets.")
except Exception as e:
    print("⚠️  No se encontró la key en Secrets. Pégala aquí solo")
    print("   temporalmente (bórrala después de ejecutar):")
    USDA_API_KEY = input("Pega tu API key de USDA: ").strip()

✅ API key cargada correctamente desde Secrets.


## Celda 2 — Consultar la API de USDA FoodData Central

In [6]:
# ══════════════════════════════════════════════════════════════════
# CELDA 2 (VERSIÓN HÍBRIDA) — Búsqueda por texto + IDs exactos
# para casos problemáticos
# ══════════════════════════════════════════════════════════════════

import requests
import time

BASE_URL = "https://api.nal.usda.gov/fdc/v1"

INGREDIENTES = {
    "manzana":   "apple, raw",
    "zanahoria": "carrots, raw",
    "limon":     "lemons, raw",
    "naranja":   "oranges, raw",
    "pimiento":  "peppers, sweet, red, raw",
    "tomate":    "tomatoes, red, ripe, raw",
}

# ⚠️ RELLENA AQUÍ el FDC ID que encuentres en fdc.nal.usda.gov
# Para los ingredientes en este diccionario, se usará el ID
# directamente en lugar de buscar por texto.
IDS_MANUALES = {
    "limon": 167746,
}

PALABRAS_EXCLUIDAS = [
    "juice", "peel", "rose", "dried", "canned", "cooked", "boiled",
    "frozen", "sauce", "powder", "extract", "concentrate", "candied",
    "sweetened", "syrup", "grass", "citronella", "leaves", "leaf",
]

def obtener_por_id(fdc_id, api_key):
    """Obtiene un alimento directamente por su FDC ID."""
    url = f"{BASE_URL}/food/{fdc_id}"
    params = {"api_key": api_key}
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    return resp.json()


def buscar_alimento(query, api_key, nombre_es):
    url = f"{BASE_URL}/foods/search"
    params = {
        "query": query,
        "api_key": api_key,
        "dataType": ["Foundation", "SR Legacy"],
        "pageSize": 20,
    }
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    data = resp.json()

    candidatos = data.get("foods", [])
    if not candidatos:
        return None

    candidatos_validos = [
        c for c in candidatos
        if not any(palabra in c.get("description", "").lower()
                   for palabra in PALABRAS_EXCLUIDAS)
    ]

    if not candidatos_validos:
        return candidatos[0]

    return candidatos_validos[0]


def extraer_nutrientes(food_item):
    nutrientes_buscados = {
        "Energy": "calorias_kcal",
        "Protein": "proteinas_g",
        "Carbohydrate, by difference": "carbohidratos_g",
        "Total lipid (fat)": "grasas_g",
        "Fiber, total dietary": "fibra_g",
    }
    resultado = {v: None for v in nutrientes_buscados.values()}
    for nutriente in food_item.get("foodNutrients", []):
        # La estructura difiere ligeramente entre /search y /food/{id}
        nombre = (
            nutriente.get("nutrientName")
            or nutriente.get("nutrient", {}).get("name", "")
        )
        valor = nutriente.get("value") or nutriente.get("amount")
        for clave_usda, clave_es in nutrientes_buscados.items():
            if clave_usda.lower() in nombre.lower():
                resultado[clave_es] = valor
    return resultado


print("Consultando USDA FoodData Central (modo híbrido)...\n")

resultados_usda = {}

for nombre_es, query in INGREDIENTES.items():
    print(f"  Buscando: {nombre_es} ({query})...")

    if nombre_es in IDS_MANUALES and IDS_MANUALES[nombre_es]:
        # Usar ID exacto
        food = obtener_por_id(IDS_MANUALES[nombre_es], USDA_API_KEY)
        print(f"    (usando FDC ID manual: {IDS_MANUALES[nombre_es]})")
    else:
        # Buscar por texto
        food = buscar_alimento(query, USDA_API_KEY, nombre_es)

    if food is None:
        print(f"    ⚠️  No se encontró '{query}'")
        continue

    nutrientes = extraer_nutrientes(food)
    resultados_usda[nombre_es] = {
        "fdc_id": food.get("fdcId"),
        "descripcion_usda": food.get("description"),
        **nutrientes
    }
    print(f"    ✅ Encontrado: {food.get('description')} (FDC ID: {food.get('fdcId')})")

    time.sleep(0.5)

print("\n✅ Consulta completada.")

Consultando USDA FoodData Central (modo híbrido)...

  Buscando: manzana (apple, raw)...
    ✅ Encontrado: Apples, fuji, with skin, raw (FDC ID: 1750340)
  Buscando: zanahoria (carrots, raw)...
    ✅ Encontrado: Carrots, raw (FDC ID: 170393)
  Buscando: limon (lemons, raw)...
    (usando FDC ID manual: 167746)
    ✅ Encontrado: Lemons, raw, without peel (FDC ID: 167746)
  Buscando: naranja (oranges, raw)...
    ✅ Encontrado: Oranges, raw, Florida (FDC ID: 169918)
  Buscando: pimiento (peppers, sweet, red, raw)...
    ✅ Encontrado: Peppers, sweet, red, raw (FDC ID: 170108)
  Buscando: tomate (tomatoes, red, ripe, raw)...
    ✅ Encontrado: Tomatoes, red, ripe, raw, year round average (FDC ID: 170457)

✅ Consulta completada.


In [8]:
# ══════════════════════════════════════════════════════════════════
# FUNCIÓN extraer_nutrientes CORREGIDA — filtra por unidad para
# evitar confundir kcal con kJ
# Sustituye SOLO la función extraer_nutrientes en tu Celda 2
# (deja el resto del código igual) y vuelve a ejecutar la celda
# completa para regenerar resultados_usda con los valores correctos
# ══════════════════════════════════════════════════════════════════

def extraer_nutrientes(food_item):
    nutrientes_buscados = {
        "Energy": "calorias_kcal",
        "Protein": "proteinas_g",
        "Carbohydrate, by difference": "carbohidratos_g",
        "Total lipid (fat)": "grasas_g",
        "Fiber, total dietary": "fibra_g",
    }
    resultado = {v: None for v in nutrientes_buscados.values()}

    for nutriente in food_item.get("foodNutrients", []):
        nombre = (
            nutriente.get("nutrientName")
            or nutriente.get("nutrient", {}).get("name", "")
        )
        valor = nutriente.get("value")
        if valor is None:
            valor = nutriente.get("amount")

        unidad = (
            nutriente.get("unitName")
            or nutriente.get("nutrient", {}).get("unitName", "")
        )
        unidad = (unidad or "").upper()

        for clave_usda, clave_es in nutrientes_buscados.items():
            if clave_usda.lower() not in nombre.lower():
                continue

            # Caso especial: "Energy" tiene versiones en KCAL y KJ.
            # Solo aceptamos la que está en KCAL.
            if clave_usda == "Energy" and unidad != "KCAL":
                continue

            resultado[clave_es] = valor

    return resultado


# ── Volver a procesar los 6 ingredientes con la función corregida ──
# (reutiliza 'resultados_usda' ya obtenido, solo recalcula calorías)

import requests

def obtener_por_id(fdc_id, api_key):
    url = f"{BASE_URL}/food/{fdc_id}"
    resp = requests.get(url, params={"api_key": api_key})
    resp.raise_for_status()
    return resp.json()

print("Recalculando valores nutricionales con el filtro de unidad corregido...\n")

for nombre_es in resultados_usda:
    fdc_id = resultados_usda[nombre_es]["fdc_id"]
    food_completo = obtener_por_id(fdc_id, USDA_API_KEY)
    nutrientes_corregidos = extraer_nutrientes(food_completo)
    resultados_usda[nombre_es].update(nutrientes_corregidos)
    print(f"  {nombre_es}: {nutrientes_corregidos['calorias_kcal']} kcal")

print("\n✅ Valores corregidos.")

import pandas as pd
df_usda = pd.DataFrame.from_dict(resultados_usda, orient="index")
print("\nTabla actualizada:\n")
print(df_usda.to_string())

Recalculando valores nutricionales con el filtro de unidad corregido...

  manzana: 58.20306 kcal
  zanahoria: 41.0 kcal
  limon: 29.0 kcal
  naranja: 46.0 kcal
  pimiento: 26.0 kcal
  tomate: 18.0 kcal

✅ Valores corregidos.

Tabla actualizada:

            fdc_id                              descripcion_usda  calorias_kcal  proteinas_g  carbohidratos_g  grasas_g  fibra_g
manzana    1750340                  Apples, fuji, with skin, raw       58.20306     0.148438        15.651162    0.1625    2.075
zanahoria   170393                                  Carrots, raw       41.00000     0.930000         9.580000    0.2400    2.800
limon       167746                     Lemons, raw, without peel       29.00000     1.100000         9.320000    0.3000    2.800
naranja     169918                         Oranges, raw, Florida       46.00000     0.700000        11.540000    0.2100    2.400
pimiento    170108                      Peppers, sweet, red, raw       26.00000     0.990000         6.03000

## Celda 3 — Mostrar resultados crudos de la API

In [9]:
import pandas as pd

df_usda = pd.DataFrame.from_dict(resultados_usda, orient="index")
print("Datos nutricionales obtenidos de USDA FoodData Central:\n")
print(df_usda.to_string())

Datos nutricionales obtenidos de USDA FoodData Central:

            fdc_id                              descripcion_usda  calorias_kcal  proteinas_g  carbohidratos_g  grasas_g  fibra_g
manzana    1750340                  Apples, fuji, with skin, raw       58.20306     0.148438        15.651162    0.1625    2.075
zanahoria   170393                                  Carrots, raw       41.00000     0.930000         9.580000    0.2400    2.800
limon       167746                     Lemons, raw, without peel       29.00000     1.100000         9.320000    0.3000    2.800
naranja     169918                         Oranges, raw, Florida       46.00000     0.700000        11.540000    0.2100    2.400
pimiento    170108                      Peppers, sweet, red, raw       26.00000     0.990000         6.030000    0.3000    2.100
tomate      170457  Tomatoes, red, ripe, raw, year round average       18.00000     0.880000         3.890000    0.2000    1.200


## Celda 4 — Combinar con huella de carbono (Apéndice A del TFM) y riesgo de desperdicio

In [10]:
co2_apendice_a = {
    "manzana":   0.43,
    "zanahoria": 0.43,
    "limon":     0.40,
    "naranja":   0.45,
    "pimiento":  0.53,
    "tomate":    1.44,
}

riesgo_desperdicio = {nombre: "Alto" for nombre in INGREDIENTES}

tabla_maestra_final = []
for nombre in INGREDIENTES:
    if nombre not in resultados_usda:
        continue
    fila = {
        "nombre": nombre,
        **resultados_usda[nombre],
        "co2_kg_por_kg": co2_apendice_a[nombre],
        "riesgo_desperdicio": riesgo_desperdicio[nombre],
    }
    tabla_maestra_final.append(fila)

df_tabla_maestra = pd.DataFrame(tabla_maestra_final)

print("\n" + "═" * 70)
print("  TABLA MAESTRA FINAL — datos reales USDA + CO2 (Apéndice A)")
print("═" * 70)
columnas_mostrar = [
    "nombre", "descripcion_usda", "calorias_kcal", "proteinas_g",
    "carbohidratos_g", "grasas_g", "fibra_g",
    "co2_kg_por_kg", "riesgo_desperdicio"
]
print(df_tabla_maestra[columnas_mostrar].to_string(index=False))


══════════════════════════════════════════════════════════════════════
  TABLA MAESTRA FINAL — datos reales USDA + CO2 (Apéndice A)
══════════════════════════════════════════════════════════════════════
   nombre                             descripcion_usda  calorias_kcal  proteinas_g  carbohidratos_g  grasas_g  fibra_g  co2_kg_por_kg riesgo_desperdicio
  manzana                 Apples, fuji, with skin, raw       58.20306     0.148438        15.651162    0.1625    2.075           0.43               Alto
zanahoria                                 Carrots, raw       41.00000     0.930000         9.580000    0.2400    2.800           0.43               Alto
    limon                    Lemons, raw, without peel       29.00000     1.100000         9.320000    0.3000    2.800           0.40               Alto
  naranja                        Oranges, raw, Florida       46.00000     0.700000        11.540000    0.2100    2.400           0.45               Alto
 pimiento                     P

## Celda 5 — Guardar y descargar la tabla maestra

In [11]:
from google.colab import files

ruta_csv = "tabla_maestra_usda_real.csv"
df_tabla_maestra.to_csv(ruta_csv, index=False, encoding="utf-8-sig")

print(f"\n✅ Tabla maestra guardada en: {ruta_csv}")
files.download(ruta_csv)


✅ Tabla maestra guardada en: tabla_maestra_usda_real.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>